In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    """Find repo root by walking upward until expected project markers are found."""
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / "config").exists() and (path / "notebooks").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find repo root. Launch notebook from inside the repository root.")

REPO_ROOT = find_repo_root()

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PREPROCESSING_DATA_DIR = REPO_ROOT / "data" / "processed" / "preprocessing"
OBJ2_DATA_DIR = REPO_ROOT / "data" / "processed" / "objective2_demand"
OBJ2_OUTPUT_DIR = REPO_ROOT / "outputs" / "objective2_demand"
OBJ2_MODEL_DIR = OBJ2_OUTPUT_DIR / "models"
OBJ2_VALIDATION_DIR = OBJ2_OUTPUT_DIR / "validation"
CONFIG_DIR = REPO_ROOT / "config"
CALENDAR_DIR = CONFIG_DIR / "calendars"
LOOKUP_DIR = CONFIG_DIR / "lookups"

for path in [OBJ2_DATA_DIR, OBJ2_OUTPUT_DIR, OBJ2_MODEL_DIR, OBJ2_VALIDATION_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
# Task 6 input/output paths
FES_PATH = PREPROCESSING_DATA_DIR / "fes_demand_annual_2030_2045.parquet"
CALENDAR_HOURLY_PATH = PREPROCESSING_DATA_DIR / "calendar_hourly_2010_2045.parquet"
FUTURE_DAILY_CLIMATE_ONLY_PATH = OBJ2_DATA_DIR / "future_daily_demand_climate_only.parquet"
FUTURE_DAILY_FES_ANCHORED_PATH = OBJ2_DATA_DIR / "future_daily_demand_fes_anchored.parquet"


In [ ]:
future_daily_climate_only = pd.read_parquet(
    FUTURE_DAILY_CLIMATE_ONLY_PATH
).copy()

fes = pd.read_parquet(FES_PATH).copy()

# -------------------------------------------------------------------
# Basic type standardisation
# -------------------------------------------------------------------

future_daily_climate_only["date"] = pd.to_datetime(future_daily_climate_only["date"])
future_daily_climate_only["year"] = future_daily_climate_only["year"].astype(int)
future_daily_climate_only["climate_member"] = future_daily_climate_only["climate_member"].astype(str)

# Avoid float32 annual-sum precision drift
future_daily_climate_only["climate_only_daily_demand_mwh"] = (
    future_daily_climate_only["climate_only_daily_demand_mwh"].astype("float64")
)

fes["year"] = fes["year"].astype(int)
fes["fes_scenario"] = fes["fes_scenario"].astype(str)
fes["annual_demand_twh"] = fes["annual_demand_twh"].astype("float64")

allowed_scenarios = ["Holistic Transition", "Electric Engagement"]

fes = fes[
    fes["fes_scenario"].isin(allowed_scenarios)
    & fes["year"].between(2030, 2045)
].copy()

assert set(fes["fes_scenario"].unique()) == set(allowed_scenarios), \
    f"FES scenarios found: {sorted(fes['fes_scenario'].unique())}"

assert fes.duplicated(["year", "fes_scenario"]).sum() == 0

# TWh to MWh
fes["annual_demand_mwh"] = fes["annual_demand_twh"] * 1_000_000.0

# -------------------------------------------------------------------
# Annual climate-only totals by year and climate member
# -------------------------------------------------------------------

climate_annual = (
    future_daily_climate_only
    .groupby(["year", "climate_member"], as_index=False)
    .agg(
        climate_only_annual_mwh=(
            "climate_only_daily_demand_mwh",
            "sum"
        )
    )
)

assert (climate_annual["climate_only_annual_mwh"] > 0).all()

# -------------------------------------------------------------------
# Apply FES annual scaling
# -------------------------------------------------------------------

anchored = (
    future_daily_climate_only
    .merge(climate_annual, on=["year", "climate_member"], how="left")
    .merge(
        fes[["year", "fes_scenario", "annual_demand_mwh", "annual_demand_twh"]],
        on="year",
        how="inner"
    )
)

anchored["fes_scale_factor"] = (
    anchored["annual_demand_mwh"] / anchored["climate_only_annual_mwh"]
).astype("float64")

anchored["anchored_daily_demand_mwh"] = (
    anchored["climate_only_daily_demand_mwh"] * anchored["fes_scale_factor"]
).astype("float64")

future_daily_anchored = anchored[
    [
        "date",
        "year",
        "fes_scenario",
        "climate_member",
        "anchored_daily_demand_mwh",
        "fes_scale_factor",
        "climate_only_daily_demand_mwh",
        "annual_demand_twh",
    ]
].copy()

future_daily_anchored.to_parquet(
    FUTURE_DAILY_FES_ANCHORED_PATH,
    index=False,
)


In [ ]:
# QA

annual_check = (
    future_daily_anchored
    .groupby(["year", "fes_scenario", "climate_member"], as_index=False)
    .agg(
        anchored_annual_mwh=("anchored_daily_demand_mwh", "sum")
    )
)

annual_check["anchored_annual_twh"] = (
    annual_check["anchored_annual_mwh"] / 1_000_000.0
)

annual_check = annual_check.merge(
    fes[["year", "fes_scenario", "annual_demand_twh"]],
    on=["year", "fes_scenario"],
    how="left",
)

annual_check["diff_twh"] = (
    annual_check["anchored_annual_twh"] - annual_check["annual_demand_twh"]
)

annual_check["diff_mwh"] = annual_check["diff_twh"] * 1_000_000.0

print(annual_check["diff_twh"].abs().describe())
print("Max absolute difference in MWh:", annual_check["diff_mwh"].abs().max())

assert annual_check["annual_demand_twh"].notna().all()
assert annual_check["diff_twh"].abs().max() < 1e-9